<a href="https://www.kaggle.com/code/nadezhdaemelyanova/start-ml-final-project-simplerecmlp?scriptVersionId=313056760" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
!pip install fastapi==0.115.12, pandas==2.2.2, sqlalchemy==2.0.40
!pip install numpy==2.0.2, catboost==1.2.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 104.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.47
    Uninstalling SQLAlchemy-2.0.47:
      Successfully uninstalled SQLAlchemy-2.0.47
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.133.0
    Uninstalling fastapi-0.133.0:
      Successfully uninstalled fastapi-0.133.0
ERROR: pip's dependency r

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm
from collections import defaultdict
from sqlalchemy import create_engine

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Загрузка данных

In [4]:
df = pd.read_csv('/kaggle/input/datasets/nadezhdaemelyanova/allcsv/all.csv', sep=';')
df = df.set_index('Unnamed: 0')
post_text_df = df[['post_id', 'text', 'topic']].drop_duplicates().reset_index(drop=True)

# Feature engineering
1. Делаем кластеризацию K-Means на основе эмбеддингов, полученных через TF-IDF из переменной `text`.
2. Заранее разделяем на train/val, чтобы избежать лика данных (делаем расчеты по train, не заглядываем в будущее)
3. Расчет новых фич и их вставка в датасеты

In [5]:
# работаем с переменной text

from sklearn.feature_extraction.text import TfidfVectorizer

texts = post_text_df['text'].astype(str).tolist()

vectorizer = TfidfVectorizer(
    max_features=5000, 
    stop_words='english'
)

embeddings = vectorizer.fit_transform(texts)

In [6]:
from sklearn.cluster import KMeans

if 'cluster' in df.columns:
    df = df.drop(columns=['cluster'])

kmeans = KMeans(n_clusters=8, random_state=0, n_init='auto')
cluster_labels = kmeans.fit_predict(embeddings)
post_text_df['cluster'] = cluster_labels
df = df.merge(
    post_text_df[['post_id', 'cluster']], 
    on='post_id', 
    how='left'
)
print(df['cluster'].value_counts())

cluster
5    1416572
7    1101904
4     902237
6     785805
1     559665
2     557465
0     427204
3     139703
Name: count, dtype: int64


In [7]:
# cортировка по времени
df = df.sort_values('timestamp').reset_index(drop=True)

# разделение на train/val по времени (80% / 20%)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

Train size: 4712444, Val size: 1178111


In [8]:
# ----- пользовательские признаки -----
# 1. favorite_topic: самый частый топик среди просмотренных постов
user_topic_counts = train_df.groupby(['user_id', 'topic']).size().reset_index(name='count')
user_fav_topic = user_topic_counts.loc[user_topic_counts.groupby('user_id')['count'].idxmax()]
user_fav_topic = user_fav_topic.set_index('user_id')['topic'].to_dict()

# 2. loyalty = лайки / просмотры
user_loyalty = train_df.groupby('user_id')['target'].mean().to_dict()

# 3. likes_ratio = сумма лайков пользователя / сумма лайков просмотренных постов
post_likes = train_df[train_df['target']==1].groupby('post_id').size().to_dict()
train_df['post_likes'] = train_df['post_id'].map(post_likes).fillna(0)
user_sum_likes = train_df.groupby('user_id')['target'].sum()
user_sum_post_likes = train_df.groupby('user_id')['post_likes'].sum()
user_likes_ratio = (user_sum_likes / user_sum_post_likes).fillna(0).to_dict()

# ----- признаки постов -----#
# 4. likes to views ratio = количество лайков поста / количество просмотров поста
post_views = train_df.groupby('post_id').size()
post_likes_total = train_df[train_df['target']==1].groupby('post_id').size()
post_likes_to_views = (post_likes_total / post_views).fillna(0).to_dict()

# сохраняем средние значения для новых объектов
mean_loyalty = np.mean(list(user_loyalty.values()))
mean_likes_ratio = np.mean(list(user_likes_ratio.values()))
mean_likes_to_views = np.mean(list(post_likes_to_views.values()))
most_common_topic = train_df['topic'].mode()[0]

In [9]:
# применяем к train, val и df (последнее - для загрузки в базу данных и использования в приложении)
def enrich_features(df, user_fav_topic, user_loyalty, user_likes_ratio, post_likes_to_views,
                    mean_loyalty, mean_likes_ratio, mean_likes_to_views, most_common_topic):
    df = df.copy()
    # favorite_topic
    df['favorite_topic'] = df['user_id'].map(user_fav_topic).fillna(most_common_topic)
    # loyalty
    df['loyalty'] = df['user_id'].map(user_loyalty).fillna(mean_loyalty)
    # likes_ratio
    df['likes_ratio'] = df['user_id'].map(user_likes_ratio).fillna(mean_likes_ratio)
    # likes_to_views_ratio для поста
    df['likes_to_views_ratio'] = df['post_id'].map(post_likes_to_views).fillna(mean_likes_to_views)
    return df

train_df = enrich_features(train_df, user_fav_topic, user_loyalty, user_likes_ratio,
                            post_likes_to_views, mean_loyalty, mean_likes_ratio,
                            mean_likes_to_views, most_common_topic)

val_df = enrich_features(val_df, user_fav_topic, user_loyalty, user_likes_ratio,
                            post_likes_to_views, mean_loyalty, mean_likes_ratio,
                            mean_likes_to_views, most_common_topic)

df = enrich_features(df, user_fav_topic, user_loyalty, user_likes_ratio,
                            post_likes_to_views, mean_loyalty, mean_likes_ratio,
                            mean_likes_to_views, most_common_topic)

In [ ]:
# стандартизируем числовые признаки

user_num_cols = ['age', 'loyalty', 'likes_ratio']
item_num_cols = ['likes_to_views_ratio']

scaler_user = StandardScaler()
scaler_item = StandardScaler()

train_df[user_num_cols] = scaler_user.fit_transform(train_df[user_num_cols])
val_df[user_num_cols] = scaler_user.transform(val_df[user_num_cols])
df[user_num_cols] = scaler_user.transform(df[user_num_cols])

train_df[item_num_cols] = scaler_item.fit_transform(train_df[item_num_cols])
val_df[item_num_cols] = scaler_item.transform(val_df[item_num_cols])
df[item_num_cols] = scaler_item.transform(df[item_num_cols])

In [10]:
# костыль для топика: topic_le для обучения, topic для вывода вместе с text

for d in [train_df, val_df, df]:
    d['topic_le'] = d['topic']

In [11]:
# категориальные признаки для пользователя
user_cat_cols = ['gender', 'country', 'exp_group', 'os', 'source', 'favorite_topic']
# категориальные признаки для поста
item_cat_cols = ['topic_le', 'cluster']

# LabelEncoder для каждого столбца
user_encoders = {}
item_encoders = {}

for col in user_cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    val_df[col] = le.transform(val_df[col].astype(str))
    df[col] = le.transform(df[col].astype(str))
    user_encoders[col] = le

for col in item_cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    val_df[col] = le.transform(val_df[col].astype(str))
    df[col] = le.transform(df[col].astype(str))
    item_encoders[col] = le

In [12]:
df['topic_le'].value_counts()

topic_le
3    2181587
1    1424381
5     787694
4     592024
0     370733
2     298325
6     235811
Name: count, dtype: int64

# Построение архитектуры нейросети
Для бинарной классификации взаимодействия пользователя с постом (лайк/просмотр) построен широкий многослойный перцептрон:
- Linear(входная размерность → 256) → ReLU → Dropout(0.25)
- Linear(256 → 64) → ReLU → Dropout(0.25)
- Linear(64 → 1)

Выход — логит (не сигмоида), так как используется `BCEWithLogitsLoss`, которая объединяет сигмоиду и бинарную кросс-энтропию.

Dropout (0.25) помогает бороться с переобучением.

Большой размер батча (8192) даёт более стабильную оценку градиента и ускоряет обучение на GPU + сглаживает шум при сильном дисбалансе классов.

Для скорости обучения взято всего 3 эпохи.


In [13]:
# задаем гиперпараметры

SEED = 0
BATCH_SIZE = 8192
EPOCHS = 3
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

## Входные данные
- Категориальные признаки пользователя (user_cat_cols): gender, country, exp_group, os, source, favorite_topic
- Числовые признаки пользователя (user_num_cols): age, loyalty
- Категориальные признаки поста (item_cat_cols): topic_le, cluster
- Числовые признаки поста (item_num_cols): likes_to_views_ratio

In [14]:
user_cat_cols = ['gender', 'country', 'exp_group', 'os', 'source', 'favorite_topic']
user_num_cols = ['age', 'loyalty']

item_cat_cols = ['topic_le', 'cluster']
item_num_cols = ['likes_to_views_ratio']

target_col = 'target'

In [15]:
# создаем кастомный класс датасета

class RecDataset(Dataset):
    def __init__(self, df, user_cat_cols, user_num_cols, item_cat_cols, item_num_cols, target_col='target'):
        self.user_cat = df[user_cat_cols].values.astype(np.int64)
        self.user_num = df[user_num_cols].values.astype(np.float32)
        self.item_cat = df[item_cat_cols].values.astype(np.int64)
        self.item_num = df[item_num_cols].values.astype(np.float32)
        self.y = df[target_col].values.astype(np.float32)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.user_cat[idx], dtype=torch.long),
            torch.tensor(self.user_num[idx], dtype=torch.float32),
            torch.tensor(self.item_cat[idx], dtype=torch.long),
            torch.tensor(self.item_num[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )

train_ds = RecDataset(train_df, user_cat_cols, user_num_cols, item_cat_cols, item_num_cols, target_col)
val_ds = RecDataset(val_df, user_cat_cols, user_num_cols, item_cat_cols, item_num_cols, target_col)

In [16]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

## Архитектура модели
Модель состоит из трех частей: слой эмбеддингов, конкатенация, MLP-классификатор.
1. Для каждого категориального признака создаётся `nn.Embedding`. Размер эмбеддинга вычисляется по правилу:
$$\text{emb\_dim} = \min\left(50,\; \max\left(4,\; \text{round}\left(8 \cdot \text{cardinality}^{0.25}\right)\right)\right)$$
2. Все эмбеддиги конкатенируются в один вектор. Общая размерность эмбеддингов равна сумме размерностей всех категориальных полей + добавляются числовые признаки пользователя и поста
3. Многослойный перцептрон с `ReLU` и `Dropout(0.25)` -> `logit`

In [17]:
class SimpleRecMLP(nn.Module):
    def __init__(self, cat_cardinalities, num_user, num_item, emb_dim_rule='auto', dropout=0.2):
        super().__init__()
        
        self.embeddings = nn.ModuleDict()
        total_emb_dim = 0
        
        for col, card in cat_cardinalities.items():
            if emb_dim_rule == 'auto':
                # разброс от 4 до 50 - баланс между переобучением и недообучением
                emb_dim = min(50, max(4, int(round(card ** 0.25 * 8))))
            else:
                emb_dim = emb_dim_rule
            self.embeddings[col] = nn.Embedding(card, emb_dim)

            # инициализируем маленькими случайными числами, чтобы градиенты не взрывались в начале обучения
            nn.init.normal_(self.embeddings[col].weight, std=0.01)
            total_emb_dim += emb_dim

        # конкатенация категориальных эмбеддингов с числовыми признаками юзера и поста
        input_dim = total_emb_dim + num_user + num_item
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        
    def forward(self, user_cat, user_num, item_cat, item_num):
        embs = []
        
        for i, col in enumerate(user_cat_cols):
            embs.append(self.embeddings[col](user_cat[:, i]))
        for i, col in enumerate(item_cat_cols):
            embs.append(self.embeddings[col](item_cat[:, i]))
        
        x = torch.cat(embs + [user_num, item_num], dim=1)
        
        # в BCEWithLogitsLoss кладем одномерный тензор
        logit = self.mlp(x).squeeze(1)
        return logit

In [18]:
# словарь: имя категориального признака : уникальные значения,
# для определения размерностей входных слоев эмбеддингов

cat_cardinalities = {}
for col in user_cat_cols:
    cat_cardinalities[col] = int(max(train_df[col].max(), val_df[col].max()) + 1)
for col in item_cat_cols:
    cat_cardinalities[col] = int(max(train_df[col].max(), val_df[col].max()) + 1)

cat_cardinalities

{'gender': 2,
 'country': 11,
 'exp_group': 5,
 'os': 2,
 'source': 2,
 'favorite_topic': 7,
 'topic_le': 7,
 'cluster': 8}

In [19]:
# функция потерь BCEWithLogitsLoss с балансировкой классов

pos = train_df[target_col].sum()
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos], device=device, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## Обучение, валидация, инференс и финальный замер качества


In [20]:
# вычисление loss, auc и pr-auc на валидационной выборке

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs = []
    all_y = []
    total_loss = 0.0
    n = 0
    
    for user_cat, user_num, item_cat, item_num, y in loader:
        user_cat = user_cat.to(device, non_blocking=True)
        user_num = user_num.to(device, non_blocking=True)
        item_cat = item_cat.to(device, non_blocking=True)
        item_num = item_num.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        logits = model(user_cat, user_num, item_cat, item_num)
        loss = criterion(logits, y)
        
        probs = torch.sigmoid(logits)
        
        bs = len(y)
        total_loss += loss.item() * bs
        n += bs
        
        all_probs.append(probs.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())
    
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)

    # вычисляем метрики один раз, вместо усреднения по батчам
    auc = roc_auc_score(all_y, all_probs) if len(np.unique(all_y)) > 1 else np.nan
    pr_auc = average_precision_score(all_y, all_probs) if len(np.unique(all_y)) > 1 else np.nan

    # получаем средневзвешенный loss по всей выборке
    return total_loss / n, auc, pr_auc

In [21]:
# цикл обучения

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY):
    model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=1)
    
    best_score = -1
    best_state = None
    patience = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n = 0
        
        for user_cat, user_num, item_cat, item_num, y in train_loader:
            user_cat = user_cat.to(device, non_blocking=True)
            user_num = user_num.to(device, non_blocking=True)
            item_cat = item_cat.to(device, non_blocking=True)
            item_num = item_num.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            logits = model(user_cat, user_num, item_cat, item_num)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            bs = len(y)
            running_loss += loss.item() * bs
            n += bs
        
        train_loss = running_loss / n
        val_loss, val_auc, val_pr_auc = evaluate(model, val_loader)
        
        scheduler.step(val_auc)
        
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | val_auc={val_auc:.5f} | val_pr_auc={val_pr_auc:.5f}")
        
        if val_auc > best_score:
            best_score = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print("Early stopping")
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    return model

In [22]:
model = SimpleRecMLP(
    cat_cardinalities=cat_cardinalities,
    num_user=len(user_num_cols),
    num_item=len(item_num_cols),
    dropout=0.25
)

model = train_model(model, train_loader, val_loader)

Epoch 01 | train_loss=1.18923 | val_loss=1.26640 | val_auc=0.57514 | val_pr_auc=0.15044
Epoch 02 | train_loss=1.12680 | val_loss=1.24479 | val_auc=0.58370 | val_pr_auc=0.15373
Epoch 03 | train_loss=1.11838 | val_loss=1.26671 | val_auc=0.59023 | val_pr_auc=0.15668


In [23]:
# возвращает вероятности p(target=1)

@torch.no_grad()
def predict_proba(model, df):
    ds = RecDataset(df, user_cat_cols, user_num_cols, item_cat_cols, item_num_cols, target_col)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    model.eval()
    probs = []
    
    for user_cat, user_num, item_cat, item_num, _ in loader:
        user_cat = user_cat.to(device, non_blocking=True)
        user_num = user_num.to(device, non_blocking=True)
        item_cat = item_cat.to(device, non_blocking=True)
        item_num = item_num.to(device, non_blocking=True)
        
        logits = model(user_cat, user_num, item_cat, item_num)
        probs.append(torch.sigmoid(logits).cpu().numpy())
    
    return np.concatenate(probs)

In [24]:
# расчет финальной метрики (доля пользователей, которым попали в топ-k)
# с фильтрацией по уже лайкнутым постам

def hitrate_at_k(df, probs, user_liked_posts, k=5):
    tmp = df[['user_id', 'post_id', 'target']].copy()
    tmp['prob'] = probs
    
    user_groups = tmp.groupby('user_id')
    hits = 0
    total_users = 0

    for user_id, g in user_groups:
        
        liked_set = user_liked_posts.get(user_id, set())
        g_filtered = g[~g['post_id'].isin(liked_set)]
        
        if len(g_filtered) == 0:
            total_users += 1
            continue
        
        g_sorted = g_filtered.sort_values('prob', ascending=False)
        topk = g_sorted.head(k)
        
        if topk['target'].max() > 0:
            hits += 1
        total_users += 1
    
    return hits / total_users if total_users > 0 else 0.0

In [25]:
val_probs = predict_proba(model, val_df)

# собираем историю лайков из train_df
user_liked_posts = train_df[train_df['target'] == 1].groupby('user_id')['post_id'].apply(set).to_dict()

hr5_filtered = hitrate_at_k(val_df, val_probs, user_liked_posts, k=5)
print("hitrate@5:", hr5)

hitrate@5: 0.5633823357834559


In [26]:
# сохранение модели

import torch
import joblib
import os

os.makedirs("art", exist_ok=True)

model_test = {
    "model_state_dict": model.state_dict(),
    "cat_cardinalities": cat_cardinalities,
    "feature_cols": {
        "user_cat_cols": user_cat_cols,
        "user_num_cols": user_num_cols,
        "item_cat_cols": item_cat_cols,
        "item_num_cols": item_num_cols,
    }
}

torch.save(model_test, "art/model_test.pt")

In [ ]:
# import os
# import joblib
# import torch
# import torch.nn as nn

# def get_model_path(path: str) -> str:
#     if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None:
#         return path
#     if os.environ.get("IS_LMS") == "1":
#         return "/workdir/user_input/model"
#     return path

# class SimpleRecMLP(nn.Module):
#     def __init__(self, cat_cardinalities, num_user, num_item, user_cat_cols, item_cat_cols, emb_dim_rule='auto', dropout=0.2):
#         super().__init__()
#         self.user_cat_cols = user_cat_cols
#         self.item_cat_cols = item_cat_cols
#         self.embeddings = nn.ModuleDict()
#         total_emb_dim = 0

#         for col, card in cat_cardinalities.items():
#             if emb_dim_rule == 'auto':
#                 emb_dim = min(50, max(4, int(round(card ** 0.25 * 8))))
#             else:
#                 emb_dim = emb_dim_rule
#             self.embeddings[col] = nn.Embedding(card, emb_dim)
#             nn.init.normal_(self.embeddings[col].weight, std=0.01)
#             total_emb_dim += emb_dim

#         input_dim = total_emb_dim + num_user + num_item
#         self.mlp = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(256, 64),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(64, 1)
#         )

#     def forward(self, user_cat, user_num, item_cat, item_num):
#         embs = []
#         for i, col in enumerate(self.user_cat_cols):
#             embs.append(self.embeddings[col](user_cat[:, i]))
#         for i, col in enumerate(self.item_cat_cols):
#             embs.append(self.embeddings[col](item_cat[:, i]))
#         x = torch.cat(embs + [user_num, item_num], dim=1)
#         return self.mlp(x).squeeze(1)

# checkpoint = torch.load("art/checkpoint.pt", map_location="cpu", weights_only=False)

# cols = checkpoint["feature_cols"]
# model = SimpleRecMLP(
#     cat_cardinalities=checkpoint["cat_cardinalities"],
#     num_user=len(cols["user_num_cols"]),
#     num_item=len(cols["item_num_cols"]),
#     user_cat_cols=cols["user_cat_cols"],
#     item_cat_cols=cols["item_cat_cols"],
#     dropout=0.25
# )
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()

# artifacts = {
#     "vectorizer": checkpoint["vectorizer"],
#     "kmeans": checkpoint["kmeans"],
#     "user_encoders": checkpoint["user_encoders"],
#     "item_encoders": checkpoint["item_encoders"],
#     "scaler_user": checkpoint["scaler_user"],
#     "scaler_item": checkpoint["scaler_item"],
#     "cat_cardinalities": checkpoint["cat_cardinalities"],
#     "cols": cols,
# }

In [ ]:
# cols = checkpoint["feature_cols"]
# model = SimpleRecMLP(
#     cat_cardinalities=checkpoint["cat_cardinalities"],
#     num_user=len(cols["user_num_cols"]),
#     num_item=len(cols["item_num_cols"]),
#     user_cat_cols=cols["user_cat_cols"],
#     item_cat_cols=cols["item_cat_cols"],
#     dropout=0.25
# )
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()

# uc = torch.zeros((2, len(art["cols"]["user_cat_cols"])), dtype=torch.long)
# un = torch.zeros((2, len(art["cols"]["user_num_cols"])), dtype=torch.float32)
# ic = torch.zeros((2, len(art["cols"]["item_cat_cols"])), dtype=torch.long)
# inn = torch.zeros((2, len(art["cols"]["item_num_cols"])), dtype=torch.float32)

# with torch.no_grad():
#     logits = model(uc, un, ic, inn)
#     probs = torch.sigmoid(logits)

# assert probs.shape == (2,)
# assert torch.isfinite(probs).all()
# print("Model OK:", probs.tolist())

In [ ]:
# загрузка информации о юзерах в БД

user_cols = [
    "user_id",
    "gender",
    "country",
    "exp_group",
    "os",
    "source",
    "favorite_topic",
    "age",
    "loyalty",
    "likes_ratio",
]

data_users = df.copy()[user_cols].drop_duplicates("user_id")

engine = create_engine(
    "postgresql://login:password@"
    "postgres.lab.karpov.courses:6432/startml"
)

with engine.connect() as connection:
    data_users.to_sql(
        "nadezhda01em_users_test_1",
        con=connection,
        index=False
    )

In [ ]:
# загрузка информации о постах в БД

data_posts = post_text_df.merge(
    df[["post_id", "likes_to_views_ratio", "topic_le"]].drop_duplicates("post_id"),
    on="post_id",
    how="left"
)[["post_id", "text", "topic", "topic_le", "cluster", "likes_to_views_ratio"]].drop_duplicates("post_id")

# костыль: не все посты из post_text_df попали в df
# topic_le - заменим самым частым
data_posts["topic_le"] = data_posts["topic_le"].fillna(3)
# likes_to_views_ratio - заменим тем что точно не встречалось
data_posts["likes_to_views_ratio"] = data_posts["likes_to_views_ratio"].fillna(-2)

engine = create_engine(
    "postgresql://login:password@"
    "postgres.lab.karpov.courses:6432/startml"
)

with engine.connect() as connection:
    data_posts.to_sql(
        "nadezhda01em_posts_test_1",
        con=connection,
        index=False
    )